# `(α, β)` sweep: does the reconstructed depth map always collapse to `z_middle`?

In the real production run, the reconstructed zmap converged to `z_middle` (the regularization
target) both at the initial state and at the checkpoint, with β barely moving over 150 training
iterations (~1.0e-7 throughout). `resolve_initial_α_β` is a local, unconstrained, single-starting-
point LD_LBFGS search (`src/tune_regularization.jl`) -- plausible it got stuck near its own
starting point (`α_start = β_start = 1e-7`) rather than genuinely exploring the landscape.

**The heavy computation lives in `scripts/test_reconstruction_alpha_beta.jl`, run via
`sbatch cluster_scripts/run_test_reconstruction_alpha_beta.sh`, not here** -- same reasoning as
the other convergence studies. That script sweeps `α, β ∈ {0, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4,
1e-3}` (64 pairs) against the monochromatic-lens **initial** geoms only (not the checkpoint's
optimized geoms), using the same deterministic test object/noise `reconstruction_initial.png`
already used -- a direct re-examination of the exact case that showed the collapse, not a fresh
sample. `xtol_rel = 1e-5` throughout (validated in `convergence_studies/xtol_rel_2026-11-18T17:30:44.528`
to match `xtol_rel=0`'s L to 3-4 sig figs at ~3x fewer iterations), to keep 64 reconstructions
affordable.

In [ ]:
cd(isdir("../../End2EndThermalImg.jl") ? "../../End2EndThermalImg.jl" : "../../End2EndThermalImg")
import Pkg
Pkg.activate(".")

In [ ]:
using JLD2
using PythonPlot
using DataFrames

## Load precomputed results

Picks the most recent `convergence_studies/alpha_beta_sweep_<datetime>/` run folder by default --
set `run_folder` explicitly instead if you want a specific earlier run.

In [ ]:
root = "convergence_studies"
run_names = sort(filter(name -> startswith(name, "alpha_beta_sweep_"), readdir(root)))
run_folder = joinpath(root, run_names[end])
println("Loading results from: ", run_folder)

results_file = joinpath(run_folder, "results.jld2")
source_run_folder, alphas, betas, test_object, noisy_image, summary_rows, object_opts, php, imghp =
    JLD2.load(results_file, "source_run_folder", "alphas", "betas", "test_object", "noisy_image",
              "summary_rows", "object_opts", "php", "imghp")

println("source_run_folder = ", source_run_folder)
println("alphas = ", alphas)
println("betas  = ", betas)

df = DataFrame(summary_rows)

## Summary table

In [ ]:
df

## Ground truth

The true temperature and depth maps the reconstructions below are being compared against.

In [ ]:
fig, ax = subplots(1, 2; figsize=(8, 4))
im0 = ax[0].imshow(test_object.Tmap, cmap="magma")
fig.colorbar(im0, ax=ax[0], label="T (Kelvin)")
ax[0].set_title("ground truth Tmap")
ax[0].axis("off")

im1 = ax[1].imshow(test_object.zmap, cmap="viridis")
fig.colorbar(im1, ax=ax[1], label="z (unitless)")
ax[1].set_title("ground truth zmap")
ax[1].axis("off")

fig.tight_layout()
display(fig)

## Reconstructed `zmap` across the `(α, β)` grid

Fixed color scale (the ground truth's own `zmap` extrema) across every panel: a reconstruction
that's collapsed to `z_middle` shows up as a flat, uniform color, while one that's recovered real
structure shows visible contrast matching the ground truth above. Rows are `β` (increasing
downward), columns are `α` (increasing rightward).

In [ ]:
zmap_vmin, zmap_vmax = extrema(test_object.zmap)

fig, axes = subplots(length(betas), length(alphas); figsize=(2*length(alphas), 2*length(betas)))
for (iβ, β) in enumerate(betas), (iα, α) in enumerate(alphas)
    ax = axes[iβ-1, iα-1]
    object_opt = get(object_opts, (α, β), nothing)
    if object_opt === nothing
        ax.text(0.5, 0.5, "failed", ha="center", va="center", transform=ax.transAxes)
    else
        ax.imshow(object_opt.zmap, cmap="viridis", vmin=zmap_vmin, vmax=zmap_vmax)
    end
    ax.set_xticks(Float64[])
    ax.set_yticks(Float64[])
    iβ == 1 && ax.set_title("α=$α", fontsize=9)
    iα == 1 && ax.set_ylabel("β=$β", fontsize=9)
end
fig.suptitle("Reconstructed zmap", y=1.0)
fig.tight_layout()
display(fig)

## Reconstructed `Tmap` across the `(α, β)` grid

Same layout, for `Tmap` -- to check whether a similar collapse-to-background pattern shows up for
temperature at comparable (relative) regularization strength.

In [ ]:
Tmap_vmin, Tmap_vmax = extrema(test_object.Tmap)

fig, axes = subplots(length(betas), length(alphas); figsize=(2*length(alphas), 2*length(betas)))
for (iβ, β) in enumerate(betas), (iα, α) in enumerate(alphas)
    ax = axes[iβ-1, iα-1]
    object_opt = get(object_opts, (α, β), nothing)
    if object_opt === nothing
        ax.text(0.5, 0.5, "failed", ha="center", va="center", transform=ax.transAxes)
    else
        ax.imshow(object_opt.Tmap, cmap="magma", vmin=Tmap_vmin, vmax=Tmap_vmax)
    end
    ax.set_xticks(Float64[])
    ax.set_yticks(Float64[])
    iβ == 1 && ax.set_title("α=$α", fontsize=9)
    iα == 1 && ax.set_ylabel("β=$β", fontsize=9)
end
fig.suptitle("Reconstructed Tmap", y=1.0)
fig.tight_layout()
display(fig)